# 🚜 FleetOps AI — Exploratory Data Analysis
هذا النوت بوك مستقل تمامًا (Self-contained) — يشتغل فورًا في Google Colab بدون أي تثبيت.
اضغط على كل خلية واضغط Shift+Enter بالترتيب من فوق لتحت.

## 1) توليد بيانات الأسطول (نفس البيانات المستخدمة في المشروع)

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)
rng = np.random.default_rng(42)

EQUIPMENT_TYPES = {
    "Excavator": {"fuel_base": 18, "hours_base": 8.5, "maint_interval": 250},
    "Wheel Loader": {"fuel_base": 14, "hours_base": 8.0, "maint_interval": 250},
    "Dump Truck": {"fuel_base": 22, "hours_base": 9.0, "maint_interval": 300},
    "Bulldozer": {"fuel_base": 20, "hours_base": 7.5, "maint_interval": 200},
    "Crane": {"fuel_base": 16, "hours_base": 6.5, "maint_interval": 300},
    "Grader": {"fuel_base": 15, "hours_base": 7.0, "maint_interval": 250},
}
PROJECTS = ["Riyadh Metro Ext.", "Jeddah Corniche", "NEOM Site A", "Dammam Port Rd", "Makkah Housing"]
N_EQUIPMENT, START_DATE, N_DAYS = 30, datetime(2026, 6, 1), 90

equipment_list = []
for i in range(1, N_EQUIPMENT + 1):
    eq_type = rng.choice(list(EQUIPMENT_TYPES.keys()))
    equipment_list.append({"equipment_id": f"EQ-{i:03d}", "equipment_type": eq_type,
                            "project": rng.choice(PROJECTS), "purchase_year": int(rng.integers(2016, 2024))})
equipment_df = pd.DataFrame(equipment_list)

high_downtime_units = rng.choice(equipment_df["equipment_id"], size=3, replace=False)
fuel_anomaly_unit = rng.choice([e for e in equipment_df["equipment_id"] if e not in high_downtime_units], size=1)[0]
underused_project = "Dammam Port Rd"

records = []
dates = [START_DATE + timedelta(days=d) for d in range(N_DAYS)]
for _, eq in equipment_df.iterrows():
    specs = EQUIPMENT_TYPES[eq["equipment_type"]]
    is_high_downtime = eq["equipment_id"] in high_downtime_units
    is_fuel_anomaly = eq["equipment_id"] == fuel_anomaly_unit
    hours_since_maint = int(rng.integers(0, 100))
    for day_idx, date in enumerate(dates):
        weekend_factor = 0.15 if date.weekday() == 4 else 1.0
        project_trend = max(0.4, 1.0 - (day_idx / N_DAYS) * 0.6) if eq["project"] == underused_project else 1.0
        operating_hours = max(0, rng.normal(specs["hours_base"], 1.2) * weekend_factor * project_trend)
        base_downtime_prob = 0.35 if is_high_downtime else 0.08
        downtime_hours = 0.0
        if rng.random() < base_downtime_prob:
            downtime_hours = rng.uniform(1.5, 6.0) if is_high_downtime else rng.uniform(0.5, 2.5)
            operating_hours = max(0, operating_hours - downtime_hours)
        fuel_consumption = operating_hours * specs["fuel_base"] * rng.normal(1.0, 0.07)
        if is_fuel_anomaly and day_idx > (N_DAYS - 20):
            fuel_consumption *= rng.uniform(1.35, 1.6)
        hours_since_maint += operating_hours
        maintenance_flag = 0
        if hours_since_maint >= specs["maint_interval"]:
            maintenance_flag = 1
            hours_since_maint = 0
            downtime_hours += rng.uniform(2, 5)
        records.append({"date": date.strftime("%Y-%m-%d"), "equipment_id": eq["equipment_id"],
                         "equipment_type": eq["equipment_type"], "project": eq["project"],
                         "operating_hours": round(operating_hours, 2), "downtime_hours": round(downtime_hours, 2),
                         "fuel_consumption_l": round(max(fuel_consumption, 0), 2), "maintenance_flag": maintenance_flag})

daily_df = pd.DataFrame(records)
df = daily_df.merge(equipment_df, on=["equipment_id", "equipment_type", "project"])
print(f"✅ تم توليد {len(df)} سجل لـ {equipment_df.shape[0]} معدة")
df.head()

## 2) نظرة عامة سريعة على البيانات (EDA)

In [ ]:
print(df.describe(include='all'))
print("\nأنواع المعدات:\n", df['equipment_type'].value_counts())
print("\nعدد المعدات لكل مشروع:\n", df.groupby('project')['equipment_id'].nunique())

## 3) رسم: أعلى المعدات من حيث التوقف (Downtime)

In [ ]:
import matplotlib.pyplot as plt

agg = df.groupby(['equipment_id', 'equipment_type']).agg(
    operating_hours=('operating_hours', 'sum'), downtime_hours=('downtime_hours', 'sum')).reset_index()
agg['downtime_rate_%'] = (agg['downtime_hours'] / (agg['operating_hours'] + agg['downtime_hours']) * 100).round(1)
top = agg.sort_values('downtime_rate_%', ascending=False).head(8)

plt.figure(figsize=(9, 4.5))
plt.bar(top['equipment_id'], top['downtime_rate_%'], color='#e76f51')
plt.ylabel('Downtime Rate (%)')
plt.title('Top 8 Equipment by Downtime Rate')
plt.show()

## 4) رسم: معدل الاستخدام حسب المشروع

In [ ]:
proj = df.groupby('project').agg(operating_hours=('operating_hours', 'sum'), downtime_hours=('downtime_hours', 'sum')).reset_index()
proj['utilization_%'] = (proj['operating_hours'] / (proj['operating_hours'] + proj['downtime_hours']) * 100).round(1)

plt.figure(figsize=(9, 4.5))
plt.barh(proj['project'], proj['utilization_%'], color='#1a3a5c')
plt.xlabel('Utilization Rate (%)')
plt.title('Fleet Utilization by Project')
plt.xlim(0, 100)
plt.show()

## 5) تحميل البيانات كملف CSV (اختياري)
شغّل الخلية دي لو عايز تنزّل البيانات على جهازك من كولاب.

In [ ]:
from google.colab import files
df.to_csv('fleet_daily_logs.csv', index=False)
files.download('fleet_daily_logs.csv')